In [ ]:
from ultralytics import YOLO
import cv2
import cvzone
import os
import sys

weights_path_nano = "yolov8n.pt"  
weights_path_small = "yolov8s.pt"
weights_path_large = "yolov8l.pt"

video_cam_path = os.path.join("..","Videos","People.mp4")
cam = cv2.VideoCapture(video_cam_path)

user_input = input("-- Enter your Preferrence of Model Weights ( Nano / Large / Small ) : ").strip().title()
try:
    if user_input == "Nano":
        weights = weights_path_nano
    elif user_input == "Large":
        weights = weights_path_large
    elif user_input == "Small":
        weights = weights_path_small    
    else:
        print(f"Invalid Input")
except BaseException as e:
    print(f"Invalid Input")

my_model = YOLO(weights)
classnames = ["person","bicycle","car","motorbike","aeroplane","bus","train","truck","boat",
              "traffic light","fire hydrant","stop sign","parking meter", "bench", "bird", "cat",
              "dog","horse","sheep","cow","elephant","bear","zebra","giraffe","backpack","umbrella",
              "handbag","tie","suitcase","frisbee","skis","snowboard","sports ball","kite","baseball bat",
              "baseball glove","skateboard","surfboard","tennis racket","bottle","wine glass","cup","fork",
              "knife","spoon","bowl","banana","apple","sandwhich","orange","broccoli","carrot","hot dog","pizza",
              "donut","cake","chair","sofa","pottedplant","bed","diningtable","toilet","tvmonitor","laptop","mouse",
              "remote","keyboard","cell phone", "microwave", "oven", "toaster", "sink", "refridgerator", "book","clock",
              "vase", "scissors", "teddybear", "hair drier","toothbrush","paper"]

counted_people = set()

print(f"Number of Classes = {len(classnames)} | Current Model Weights = {user_input.title()} Weights")
while cam.isOpened():
    success, image = cam.read()
    if not success:
        break
    results = my_model.track(image, stream=True, persist=True)

    for r in results:
        boxes = r.boxes
        for box in boxes:

            if box.id is None:
                continue

            track_id = int(box.id[0])
            x1,y1,x2,y2 = box.xyxy[0]
            x1,y1,x2,y2 = int(x1), int(y1), int(x2), int(y2)
            
            w,h = x2-x1, y2-y1
            conf = box.conf[0]

            cls = int(box.cls[0])
            currentClass = classnames[cls]

            if currentClass == "person" and conf > 0.5:

                if track_id not in counted_people:
                    counted_people.add(track_id)

                cvzone.cornerRect(image,(x1,y1,w,h), t=3, colorC=(134,255,256), l=5)
                cvzone.putTextRect(image, f"Id = {int(track_id)} | Confidence = {conf:.2f} | Class = {currentClass.title()}", (max(0,x1),max(40,y1)), thickness=2, scale=0.8, offset=3)  
    
    cvzone.putTextRect(image, f"Total People = {len(counted_people)}", (50, 50), scale=1.5, thickness=2, offset=10)  

    cv2.imshow(mat=image, winname="People Detection")
    cv2.waitKey(1)
    
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cam.release()
cv2.destroyAllWindows()   
print(f"\nTotal People Counted = {len(counted_people)}") 

ModuleNotFoundError: No module named 'Weights'